# 1-2. 에이전트란 무엇인가? — Eligibility Checker 구현

**exercise 1 | 기초 개념 × APD 실습**

---

## 학습 목표
1. AI 에이전트의 정의 이해
2. 에이전트 시스템의 핵심 구성요소(모델, 도구, 메모리, 오케스트레이션) 파악
3. **시스템 프롬프트**로 에이전트의 역할과 제약 정의하기
4. **Google Gen AI SDK**로 동아리 지원자 자격 검사 에이전트 구현

---

## 이번 exercise 위치: APD 파이프라인

```
INPUT(지원서) → [Task 0: 수집] → [Task 1b: 자격검사 ← 우리가 만들 에이전트] → [Task 2: 선발] → ...
```

이번 exercise에서는 **Agentic Process Diagram(APD)** 의 Task 1b에 해당하는
**Eligibility Checker** 에이전트를 구현합니다.

> **Eligibility Checker**: 지원자의 학년, 전공, 이수 과목을 검토하여 지원 자격을 자동으로 판정합니다.

## 1. AI 에이전트란?

### 1.1 정의

> **AI 에이전트** = 환경을 인식하고, 그 환경에서 목표를 달성하기 위해 행동할 수 있는 시스템

Eligibility Checker는 전형적인 에이전트입니다:
- **지각(Perception)**: 지원서 내용을 읽어 들임
- **추론(Reasoning)**: 각 기준에 맞는지 판단
- **행동(Action)**: PASS/FAIL 판정 및 사유 출력

### 1.2 핵심 구성요소 미리보기

| 구성요소 | 역할 | Eligibility Checker에서의 역할 |
|---------|------|--------------------------------|
| **모델 (Model)** | 추론 엔진 | 지원 정보를 이해하고 기준에 맞는지 판단 |
| **도구 (Tools)** | 외부 연결 | (Week 2에서 추가) DB 조회, 규칙 검사 함수 |
| **메모리 (Memory)** | 정보 저장 | 여러 지원자 검사 결과를 대화 히스토리로 유지 |
| **오케스트레이션(orchestration)** | 흐름 조율 | 지원서 수신 → 기준 검사 → 판정 출력 순서 제어 |

## 2. APD와 동아리 모집 시나리오

### 2.1 Agentic Process Diagram(APD)이란?

APD는 AI 에이전트가 실제 업무 프로세스에서 어떻게 역할을 맡는지 표현하는 다이어그램입니다.
각 박스는 하나의 **Task**이고, 박스 안에 **수행 주체(Human / AI)와 역할(role)** 이 명시됩니다.

| 색상 | 역할 |
|------|------|
| 🟦 파랑 | 정보 수집 (Info Acquisition) |
| 🟧 주황 | 정보 분석 (Info Analysis) |
| 🟪 보라 | 의사결정 (Decision Selection) |
| 🟩 초록 | 행동 실행 (Action Implementation) |

### 2.2 오늘의 시나리오: 대학교 AI 동아리 신입 부원 모집

```
┌─────┐ ┌────────────────┐ ┌─────┐
│ INPUT │→ | Task 0: 지원서 수집·검증 │→ │ Task 1 │
│ (지원서) │ │ AI: Application Collector🟦 │ │ (병렬) │
└─────┘ └────────────────┘ └──┬──┘
 │
 ┌────────────────────────┐
 ▼ ▼ ▼
 ┌──────────┐ ┌───────────┐ ┌──────────┐
 │Task 1a: 에세이 심사│ │Task 1b: 자격 검사 🟧│ │Task 1c: 포트폴리오 │
 │AI: Essay Analyzer │ │AI:Eligibility Checker│ │AI:Portfolio Eval. │
 └──────────┘ └───────────┘ └──────────┘
```

### 2.3 Eligibility Checker의 역할

- **기능**: 지원자의 학년 / 전공 / 이수 과목이 동아리 자격 기준에 맞는지 검사
- **입력**: 지원서 데이터 (이름, 학년, 전공, 이수 과목)
- **출력**: PASS 또는 FAIL + 상세 사유
- **APD 분류**: 🟧 정보 분석 (Info Analysis) — 규칙 기반 판정

## 3. 환경 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install -q google-genai

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.colab import auth
import json
import time

# Vertex AI 인증 (Google 계정으로 로그인)
auth.authenticate_user()

PROJECT_ID = "scientific-management-494205"
LOCATION = "global"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1")
)
print('✅ 환경 설정 완료')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 708.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
✅ 환경 설정 완료


## 4. 핵심 구성요소 맛보기 — 동아리 모집 맥락으로

In [2]:
# ① 모델: 지원서 내용을 이해하고 분석
sample_application = '이름: 이지은, 학년: 2학년, 전공: 컴퓨터공학과, 이수과목: 파이썬 기초'
response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=f'다음 지원서를 한 문장으로 요약해줘: {sample_application}')
print('① 모델:', response.text.strip())

# ② 도구: 자격 기준 검사 함수 (이번 코드는 맛보기, exercise 2에서 본격적으로 다룸)
def check_grade_level(year: int) -> str:
    '''학년이 1~3학년인지 확인합니다.'''
    return '✅ 합격' if 1 <= year <= 3 else '❌ 불합격'

print('② 도구 (학년 검사):', check_grade_level(2), '← 2학년')
print('② 도구 (학년 검사):', check_grade_level(4), '← 4학년')

① 모델: 이지은은 컴퓨터공학과 2학년 학생으로, 파이썬 기초 과목을 이수했습니다.
② 도구 (학년 검사): ✅ 합격 ← 2학년
② 도구 (학년 검사): ❌ 불합격 ← 4학년


In [3]:
# ③ 메모리: 이전 검사 결과를 기억
chat = client.chats.create(model='gemini-2.5-flash-lite')
chat.send_message('이지은 학생(2학년, 컴퓨터공학과)을 PASS로 판정했어.')
mem_response = chat.send_message('이지은 학생 결과가 어땠지?')
print('③ 메모리:', mem_response.text.strip()[:80])

③ 메모리: 이지은 학생(2학년, 컴퓨터공학과)은 **PASS**로 판정되었습니다.


In [4]:
# ④ 오케스트레이션: 요청 의도 파악 → 처리 방식 결정 → 실행
ELIGIBILITY_TASKS = {
    '자격검사': '지원자 정보를 받아 자격 기준 검사',
    '기준조회': '현재 지원 자격 기준 안내',
    '결과조회': '이전 검사 결과 조회',
}

def eligibility_orchestrator(user_input: str) -> None:
    print(f"\n{'='*45}")
    print(f'📥 입력: {user_input}')

    intent_prompt = f'''다음 요청을 분석하여 JSON으로만 답하세요.
가능한 intent 값: 자격검사, 기준조회, 결과조회, 기타
문장: {user_input}
응답 형식 예시: {{"intent": "자격검사", "parameter": "지원자 정보"}}'''

    raw = client.models.generate_content(model='gemini-2.5-flash-lite', contents=intent_prompt).text.strip()
    raw = raw.replace('```json', '').replace('```', '').strip()
    try:
        parsed = json.loads(raw)
        intent = parsed.get('intent', '기타')
        parameter = parsed.get('parameter')
    except Exception:
        intent, parameter = '기타', None

    print(f"🧠 [Step 1] 의도 파악: intent='{intent}', parameter='{parameter}'")

    task_desc = ELIGIBILITY_TASKS.get(intent)
    if task_desc:
        print(f"🔧 [Step 2] 도구 선택: '{intent}' → {task_desc}")
    else:
        print(f"🔧 [Step 2] 도구 선택: 해당 도구 없음 → 모델 직접 응답")

    result = client.models.generate_content(model='gemini-2.5-flash-lite', contents=user_input).text.strip()[:120]
    print(f'⚙️  [Step 3] 실행 결과: {result}')

print('\n④ 오케스트레이션 — 단계별 처리 과정:')
eligibility_orchestrator('이지은 학생 자격 검사해줘 (2학년, 컴퓨터공학과, 파이썬기초 이수)')
eligibility_orchestrator('동아리 지원 자격 기준이 뭐야?')


④ 오케스트레이션 — 단계별 처리 과정:

📥 입력: 이지은 학생 자격 검사해줘 (2학년, 컴퓨터공학과, 파이썬기초 이수)
🧠 [Step 1] 의도 파악: intent='자격검사', parameter='지원자 정보'
🔧 [Step 2] 도구 선택: '자격검사' → 지원자 정보를 받아 자격 기준 검사
⚙️  [Step 3] 실행 결과: 이지은 학생의 정보를 확인했습니다.

제공해주신 내용은 다음과 같습니다:
*   **이름:** 이지은
*   **학년:** 2학년
*   **학과:** 컴퓨터공학과
*   **이수 과목:** 파이썬기초

다만, '자

📥 입력: 동아리 지원 자격 기준이 뭐야?
🧠 [Step 1] 의도 파악: intent='기준조회', parameter='자격 기준'
🔧 [Step 2] 도구 선택: '기준조회' → 현재 지원 자격 기준 안내
⚙️  [Step 3] 실행 결과: 동아리 지원 자격 기준은 동아리의 종류, 성격, 소속된 기관(학교, 회사, 지역사회 등)에 따라 매우 다양합니다. 하지만 일반적으로 다음과 같은 기준들이 적용될 수 있습니다.

**1. 기본적인 공통 자격:**

*


## 5. 모델 선택: 작업에 맞는 모델 고르기

Eligibility Checker는 **규칙 기반 판정** 작업입니다.
복잡한 추론보다 **빠르고 일관된 판단**이 중요합니다.

**핵심 원칙**: 모든 작업에 가장 큰 모델을 쓰는 것이 좋은 것이 아닙니다.
작업 복잡도에 맞는 모델을 선택하세요.

In [5]:
test_prompt = '지원자 이지은 (2학년, 컴퓨터공학과, 파이썬기초 이수)에게 자격 심사 통과 안내 메시지를 2문장으로 작성해줘.'

models_to_test = [
    ('gemini-2.5-flash-lite', '경량 모델 (빠름, 규칙 판정에 적합)'),
]

for model_name, description in models_to_test:
    try:
        start = time.time()
        response = client.models.generate_content(model=model_name, contents=test_prompt)
        elapsed = time.time() - start
        print(f"\n{'='*50}")
        print(f'🔷 {description} ({model_name})')
        print(f'⏱️  응답 시간: {elapsed:.2f}초')
        print(f'💬 응답: {response.text}')
    except Exception as e:
        print(f'⚠️  {model_name}: {e}')


🔷 기본 모델 (빠름, 규칙 판정에 적합) (gemini-2.5-flash)
⏱️  응답 시간: 12.03초
💬 응답: 이지은 2학년 컴퓨터공학과 지원자님의 자격 심사가 성공적으로 통과되었음을 안내드립니다. 파이썬기초 이수 경력 등 제출하신 정보를 바탕으로 좋은 결과를 얻으신 것을 진심으로 축하드립니다.


## 6. 핵심 구현: 시스템 프롬프트로 Eligibility Checker 역할 부여

에이전트의 **역할, 기준, 응답 형식**을 시스템 프롬프트에 정의합니다.
시스템 프롬프트가 잘 설계될수록 판정의 일관성이 높아집니다.

### 설계 원칙
1. **역할 선언**: 에이전트가 무엇을 하는지 명확히
2. **기준 명시**: 검사할 자격 기준을 구체적으로 나열
3. **응답 형식**: 출력 형식을 고정해 일관성 확보
4. **제약 조건**: 불확실한 경우 처리 방식 명시

In [6]:
ELIGIBILITY_SYSTEM_PROMPT = '''
당신은 대학교 AI 동아리의 신입 부원 자격을 검사하는 Eligibility Checker 에이전트입니다.

[자격 기준]
1. 학년: 1~3학년만 지원 가능 (4학년은 지원 불가)
2. 전공: 공과대학 또는 자연과학대학 소속만 지원 가능
   - 공과대학 해당: 컴퓨터공학과, 전기공학과, 기계공학과, 화학공학과, 소프트웨어학과 등
   - 자연과학대학 해당: 수학과, 물리학과, 화학과, 통계학과, 생명과학과 등
3. 이수 과목: 다음 중 최소 1개 이수 완료
   - 파이썬 기초
   - 선형대수

[응답 형식]
각 기준을 순서대로 확인한 후 최종 판정을 내려주세요:
- 학년 기준: ✅ 합격 또는 ❌ 불합격 (이유)
- 전공 기준: ✅ 합격 또는 ❌ 불합격 (이유)
- 이수 과목: ✅ 합격 또는 ❌ 불합격 (이유)
- 최종 판정: **PASS** 또는 **FAIL** — (종합 사유)

[제약]
- 모든 기준을 명시적으로 확인하세요.
- 정보가 부족한 경우 "정보 부족"으로 처리하고 FAIL 판정하세요.
- 추측하지 마세요.
'''

print('✅ 시스템 프롬프트 정의 완료')
print(f'   → 검사 기준: 학년 / 전공 / 이수 과목 (총 3가지)')

✅ 시스템 프롬프트 정의 완료
   → 검사 기준: 학년 / 전공 / 이수 과목 (총 3가지)


In [7]:
# Eligibility Checker 에이전트 생성
_eligibility_config = types.GenerateContentConfig(system_instruction=ELIGIBILITY_SYSTEM_PROMPT)

# 테스트 지원자 데이터
test_applicants = [
    {
        'name': '이지은',
        'year': 2,
        'major': '컴퓨터공학과 (공과대학)',
        'courses': ['파이썬 기초']
    },
    {
        'name': '김철수',
        'year': 4,
        'major': '경영학과 (경영대학)',
        'courses': []
    },
]

for applicant in test_applicants:
    courses_str = ', '.join(applicant['courses']) if applicant['courses'] else '없음'
    query = f'''지원자 정보:
- 이름: {applicant["name"]}
- 학년: {applicant["year"]}학년
- 전공: {applicant["major"]}
- 이수 완료 과목: {courses_str}

위 지원자의 자격을 검사해주세요.'''

    print(f"\n{'='*55}")
    print(f'📋 지원자: {applicant["name"]}')
    response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=query, config=_eligibility_config)
    print(response.text)


📋 지원자: 이지은
이지은 지원자의 자격을 검사합니다.

-   **학년 기준:** 2학년이므로 ✅ 합격
-   **전공 기준:** 컴퓨터공학과(공과대학)이므로 ✅ 합격
-   **이수 과목:** '파이썬 기초'를 이수 완료하였으므로 ✅ 합격

**최종 판정:** **PASS** — (모든 지원 자격 기준을 충족합니다.)

📋 지원자: 김철수
학년 기준: ❌ 불합격 (4학년은 지원할 수 없습니다.)
전공 기준: ❌ 불합격 (경영학과(경영대학)는 공과대학 또는 자연과학대학 소속이 아닙니다.)
이수 과목: ❌ 불합격 (필수 이수 과목(파이썬 기초, 선형대수) 중 이수한 과목이 없습니다.)
최종 판정: **FAIL** — (학년, 전공, 이수 과목 기준을 모두 충족하지 못했습니다.)


## 7. 멀티턴 대화: 여러 지원서를 연속으로 처리하기

실제 동아리 모집에서는 수십 명의 지원자를 연속으로 처리합니다.
`client.chats.create()`로 만든 `Chat` 객체를 활용하면 이전 검사 결과를 기억하며 대화를 이어갈 수 있습니다.

**컨텍스트 기억 테스트**: 마지막 질문에서 에이전트가 이전 판정을 기억하는지 확인하세요.

In [8]:
# 멀티턴 대화 — ChatSession으로 컨텍스트 유지
chat = client.chats.create(
    model='gemini-2.5-flash-lite',
    config=types.GenerateContentConfig(system_instruction=ELIGIBILITY_SYSTEM_PROMPT)
)

# 4개의 입력 — 앞 3개는 지원자 검사, 마지막은 이전 결과 확인
conversation = [
    '이지은, 2학년, 컴퓨터공학과(공과대학), 파이썬 기초 이수',
    '김철수, 4학년, 경영학과(경영대학), 이수 과목 없음',
    '박민준, 3학년, 물리학과(자연과학대학), 선형대수 이수',
    '방금 검사한 지원자 중 몇 명이 합격했어?',  # 컨텍스트 기억 테스트
]

for i, text in enumerate(conversation):
    if i < 3:
        message = f'다음 지원자를 검사해주세요: {text}'
    else:
        message = text  # 마지막은 그대로 질문

    print(f"\n{'='*55}")
    print(f'👤 입력: {message}')
    response = chat.send_message(message)
    print(f'🤖 Eligibility Checker: {response.text}')

print(f'\n📊 총 대화 턴 수: {len(chat.get_history())}')


👤 입력: 다음 지원자를 검사해주세요: 이지은, 2학년, 컴퓨터공학과(공과대학), 파이썬 기초 이수
🤖 Eligibility Checker: ## 지원자 검사 결과: 이지은

**1. 학년 기준:**
- 지원자 학년: 2학년
- 자격 기준: 1~3학년 지원 가능 (4학년 불가)
- 판정: ✅ 합격

**2. 전공 기준:**
- 지원자 전공: 컴퓨터공학과 (공과대학)
- 자격 기준: 공과대학 또는 자연과학대학 소속
- 판정: ✅ 합격

**3. 이수 과목 기준:**
- 지원자 이수 과목: 파이썬 기초
- 자격 기준: 파이썬 기초 또는 선형대수 중 최소 1개 이수 완료
- 판정: ✅ 합격

**최종 판정:** **PASS** — 모든 자격 기준을 충족합니다.

👤 입력: 다음 지원자를 검사해주세요: 김철수, 4학년, 경영학과(경영대학), 이수 과목 없음
🤖 Eligibility Checker: ## 지원자 검사 결과: 김철수

**1. 학년 기준:**
- 지원자 학년: 4학년
- 자격 기준: 1~3학년 지원 가능 (4학년 불가)
- 판정: ❌ 불합격 (4학년은 지원할 수 없습니다.)

**2. 전공 기준:**
- 지원자 전공: 경영학과 (경영대학)
- 자격 기준: 공과대학 또는 자연과학대학 소속
- 판정: ❌ 불합격 (경영대학 소속은 지원할 수 없습니다.)

**3. 이수 과목 기준:**
- 지원자 이수 과목: 없음
- 자격 기준: 파이썬 기초 또는 선형대수 중 최소 1개 이수 완료
- 판정: ❌ 불합격 (요구되는 이수 과목이 없습니다.)

**최종 판정:** **FAIL** — 학년, 전공, 이수 과목 기준을 모두 충족하지 못했습니다.

👤 입력: 다음 지원자를 검사해주세요: 박민준, 3학년, 물리학과(자연과학대학), 선형대수 이수
🤖 Eligibility Checker: ## 지원자 검사 결과: 박민준

**1. 학년 기준:**
- 지원자 학년: 3학년
- 자격 기준: 1~3학년 지원 가능 (4학년 불가)
- 판정: ✅ 합격

**2. 전공 기준:**
- 지

## 8. 에이전트 유형 실습: ReAct 패턴으로 추론 과정 시각화

**ReAct(Reasoning + Acting)** 패턴은 에이전트가 단계적으로 추론하는 과정을 명시적으로 드러냅니다.

```
Thought 1 → Action 1 → Observation 1
Thought 2 → Action 2 → Observation 2
Thought 3 → Action 3 → Observation 3
 ↓
 Final Answer
```

Eligibility Checker에 적용하면:
- **Thought**: "학년 조건을 확인해야 한다. 지원자는 3학년이다."
- **Action**: 학년 기준 검사 수행
- **Observation**: "1~3학년 기준 충족 "

단계별 추론을 보면 에이전트가 **왜** 그 판정을 내렸는지 이해할 수 있습니다.

In [16]:
def react_eligibility_checker(applicant_info: str) -> str:
    '''ReAct 패턴으로 지원자 자격을 단계적으로 검사합니다.'''
    prompt = f'''다음 지원자의 자격을 ReAct 패턴으로 단계별로 검사하세요.

[자격 기준]
- 학년: 1~3학년 (4학년 불가)
- 전공: 공과대학 또는 자연과학대학
- 이수 과목: 파이썬 기초 또는 선형대수 중 최소 1개

지원자 정보: {applicant_info}

다음 형식으로 정확히 답하세요:

Thought 1: (학년 기준 — 지원자는 몇 학년인가? 1~3학년인가?)
Action 1: 학년 기준 검사
Observation 1: (✅ 합격 또는 ❌ 불합격 + 이유)

Thought 2: (전공 기준 — 어느 대학 소속인가? 공과대학 또는 자연과학대학인가?)
Action 2: 전공 기준 검사
Observation 2: (✅ 합격 또는 ❌ 불합격 + 이유)

Thought 3: (이수 과목 — 파이썬 기초 또는 선형대수를 이수했는가?)
Action 3: 이수 과목 검사
Observation 3: (✅ 합격 또는 ❌ 불합격 + 이유)

Final Answer: **PASS** 또는 **FAIL** — (모든 기준을 종합한 최종 판정 사유)
'''
    response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=prompt)
    return response.text


# 2개의 지원자로 ReAct 패턴 테스트
test_cases = [
    '최수진, 2학년, 사회학과(사회과학대학), 파이썬 기초 이수',
    '박민준, 3학년, 물리학과(자연과학대학), 선형대수 이수',
]

for case in test_cases:
    print(f"\n{'='*55}")
    print(f'📋 지원자: {case}')
    print(react_eligibility_checker(case))


📋 지원자: 최수진, 2학년, 사회학과(사회과학대학), 파이썬 기초 이수
Thought 1: (학년 기준 — 지원자는 몇 학년인가? 1~3학년인가?)
Action 1: 학년 기준 검사
Observation 1: ✅ 합격 — 지원자는 2학년으로, 1~3학년 기준에 부합합니다.

Thought 2: (전공 기준 — 어느 대학 소속인가? 공과대학 또는 자연과학대학인가?)
Action 2: 전공 기준 검사
Observation 2: ❌ 불합격 — 지원자는 사회학과(사회과학대학) 소속으로, 공과대학 또는 자연과학대학 기준에 부합하지 않습니다.

Thought 3: (이수 과목 — 파이썬 기초 또는 선형대수를 이수했는가?)
Action 3: 이수 과목 검사
Observation 3: ✅ 합격 — 지원자는 파이썬 기초를 이수했으므로, 파이썬 기초 또는 선형대수 중 최소 1개 이수 기준에 부합합니다.

Final Answer: **FAIL** — 지원자는 사회학과(사회과학대학) 소속으로 전공 기준을 충족하지 못했습니다.

📋 지원자: 박민준, 3학년, 물리학과(자연과학대학), 선형대수 이수
Thought 1: (학년 기준 — 지원자는 몇 학년인가? 1~3학년인가?)
Action 1: 학년 기준 검사
Observation 1: ✅ 합격 - 지원자는 3학년으로, 1~3학년 기준에 부합합니다.

Thought 2: (전공 기준 — 어느 대학 소속인가? 공과대학 또는 자연과학대학인가?)
Action 2: 전공 기준 검사
Observation 2: ✅ 합격 - 지원자는 물리학과(자연과학대학) 소속으로, 공과대학 또는 자연과학대학 기준에 부합합니다.

Thought 3: (이수 과목 — 파이썬 기초 또는 선형대수를 이수했는가?)
Action 3: 이수 과목 검사
Observation 3: ✅ 합격 - 지원자는 선형대수를 이수했으므로, 파이썬 기초 또는 선형대수 중 최소 1개 이수 기준에 부합합니다.

Final Answer: **PASS** — 지원자는 학년, 전공, 이수 과목 모든

## 9. 실습 과제

### 과제 1: 자격 기준 수정 (시스템 프롬프트 설계)
아래 시스템 프롬프트에 **학점 기준(GPA 3.0 이상)** 을 추가하고 테스트해보세요.

### 과제 2: 에지 케이스 처리
다음 비정상적인 입력에서 에이전트가 어떻게 반응하는지 확인하세요:
- 정보가 부분적으로 누락된 지원서
- 경계 케이스 (1학년, 필수 과목 없음)

In [17]:
# 과제 1: 시스템 프롬프트에 학점 기준 추가
MY_ELIGIBILITY_PROMPT = '''
당신은 AI 동아리 Eligibility Checker 에이전트입니다.

[자격 기준]
1. 학년: 1~3학년
2. 전공: 공과대학 또는 자연과학대학
3. 이수 과목: 파이썬 기초 또는 선형대수 중 최소 1개
# TODO: 4번 기준을 추가하세요
# 예: 4. 학점(GPA): 3.0 이상

[응답 형식]
각 기준 확인 후: PASS 또는 FAIL + 사유
'''

_my_checker_config = types.GenerateContentConfig(system_instruction=MY_ELIGIBILITY_PROMPT)

# 과제 2: 에지 케이스 테스트
edge_cases = [
    '지원자: 이름 없음, 학년 없음, 전공 없음, 이수과목 없음',   # 정보 전혀 없음
    '지원자: 홍길동, 1학년, 컴퓨터공학과(공과대학), 이수과목 없음',  # 경계 케이스
    # TODO: 직접 에지 케이스를 추가해보세요
]

for case in edge_cases:
    print(f"\n{'='*55}")
    print(f'📋 테스트 케이스: {case}')
    response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=f'지원자 검사: {case}', config=_my_checker_config)
    print(f'🤖 판정: {response.text}')


📋 테스트 케이스: 지원자: 이름 없음, 학년 없음, 전공 없음, 이수과목 없음
🤖 판정: ## 지원자 자격 검사 결과

**지원자:** 이름 없음

**1. 학년:**
*   **결과:** FAIL
*   **사유:** 지원자의 학년 정보가 누락되었습니다. AI 동아리 Eligibility Checker는 1~3학년 재학생만 지원 가능합니다.

**2. 전공:**
*   **결과:** FAIL
*   **사유:** 지원자의 전공 정보가 누락되었습니다. AI 동아리 Eligibility Checker는 공과대학 또는 자연과학대학 소속 학생만 지원 가능합니다.

**3. 이수 과목:**
*   **결과:** FAIL
*   **사유:** 지원자의 이수 과목 정보가 누락되었습니다. AI 동아리 Eligibility Checker는 파이썬 기초 또는 선형대수 중 최소 1개 과목을 이수해야 합니다.

**4. 학점 (GPA):**
*   **결과:** FAIL
*   **사유:** 지원자의 학점 (GPA) 정보가 누락되었습니다. AI 동아리 Eligibility Checker는 3.0 이상의 학점을 보유해야 합니다.

---

**종합 결과:** 모든 기준에서 FAIL 처리되었습니다. 지원을 위해서는 누락된 정보를 모두 기입해주시기 바랍니다.

📋 테스트 케이스: 지원자: 홍길동, 1학년, 컴퓨터공학과(공과대학), 이수과목 없음
🤖 판정: ## 지원자: 홍길동

**1. 학년:** 1학년
*   **결과:** PASS
*   **사유:** 1학년은 자격 기준에 부합합니다.

**2. 전공:** 컴퓨터공학과 (공과대학)
*   **결과:** PASS
*   **사유:** 공과대학 소속이므로 자격 기준에 부합합니다.

**3. 이수 과목:** 이수 과목 없음
*   **결과:** FAIL
*   **사유:** 파이썬 기초 또는 선형대수 중 최소 1개 과목을 이수해야 하지만, 해당 지원자는 이수 과목이 없습니다.

**4. 학점 (GPA):** 정보 없음
*   **결과

## 10. 핵심 정리

| 개념 | 핵심 내용 | Eligibility Checker에서 |
|------|----------|--------------------------|
| **AI 에이전트** | 환경 인식 → 추론 → 행동 | 지원서 읽기 → 기준 판단 → PASS/FAIL 출력 |
| **핵심 구성요소** | 모델, 도구, 메모리, 오케스트레이션 | LLM이 규칙 판단, `Chat` 객체가 히스토리 유지 |
| **시스템 프롬프트** | 역할·기준·형식·제약 정의 | 자격 기준 3가지 + 응답 형식 고정 |
| **멀티턴 대화** | `Chat`(`client.chats.create()`)으로 컨텍스트 유지 | 여러 지원자 연속 처리 + 결과 집계 가능 |
| **ReAct 패턴** | Thought→Action→Observation 반복 | 학년→전공→이수과목 순서로 단계적 검사 |
| **APD** | 에이전트를 업무 프로세스에 배치 | Task 1b — 병렬 심사 단계 중 하나 |

---
**다음 exercise 예고**: 도구(Tools) & 메모리 — Application Collector, Essay Analyzer, Portfolio Evaluator 구현 🔧

## 11. exercise 3 연계: 클래스 인터페이스로 에이전트 표준화

exercise 3에서는 여러 에이전트를 하나의 파이프라인으로 연결합니다.
에이전트마다 호출 방식이 다르면 오케스트레이터 코드가 복잡해집니다.

**해결책**: 모든 에이전트가 `run()` 메서드를 갖도록 **공통 인터페이스(BaseAgentWorker)** 를 정의합니다.

```
BaseAgentWorker ← 공통 인터페이스 (추상 기본 클래스)
 └─ EligibilityCheckerAgent ← Week 1에서 만든 에이전트를 클래스로 래핑
```

> **exercise 3에서 이 클래스를 재사용합니다.**
> Colab은 노트북 간 상태 공유가 안 되므로, exercise 3 셋업 셀에서 간결하게 재정의됩니다.

In [ ]:
# ====================================================
# exercise 3 연계: BaseAgentWorker + EligibilityCheckerAgent
# ====================================================

class BaseAgentWorker:
    """APD 에이전트 워커의 공통 인터페이스.

    모든 APD 에이전트는 이 클래스를 상속하여
    동일한 run() 인터페이스를 구현합니다.
    오케스트레이터는 에이전트 유형에 관계없이 run()으로 호출할 수 있습니다.
    """
    def __init__(self, name: str, task_id: str):
        self.name = name
        self.task_id = task_id

    def run(self, input_text: str) -> str:
        """에이전트를 실행하고 결과를 반환합니다. 하위 클래스에서 구현 필수."""
        raise NotImplementedError(f'{self.__class__.__name__}.run()을 구현하세요.')

    def __repr__(self):
        return f'{self.__class__.__name__}(name="{self.name}", task="{self.task_id}")'


class EligibilityCheckerAgent(BaseAgentWorker):
    """Week 1에서 구현한 Eligibility Checker를 BaseAgentWorker 클래스로 래핑합니다.

    ELIGIBILITY_SYSTEM_PROMPT (위 셀에서 정의)를 재사용합니다.
    """

    def __init__(self):
        super().__init__(name='Eligibility Checker', task_id='Task 1b')
        self.config = types.GenerateContentConfig(
            system_instruction=ELIGIBILITY_SYSTEM_PROMPT,
        )

    def run(self, applicant_info: str) -> str:
        """지원자 정보를 받아 자격 검사 결과(PASS/FAIL + 사유)를 반환합니다.

        Args:
            applicant_info: 지원자 정보 문자열 (이름, 학년, 전공, 이수과목 포함)

        Returns:
            자격 검사 결과 텍스트
        """
        return client.models.generate_content(
            model='gemini-2.5-flash-lite', contents=applicant_info, config=self.config).text


# ====================================================
# 테스트: 클래스 인터페이스 동작 확인
# ====================================================
print('✅ BaseAgentWorker 클래스 정의 완료')
print('✅ EligibilityCheckerAgent 클래스 정의 완료\n')

# 인스턴스 생성 및 __repr__ 확인
checker_agent = EligibilityCheckerAgent()
print(f'생성된 에이전트: {checker_agent}')
print(f'  name: {checker_agent.name}')
print(f'  task_id: {checker_agent.task_id}')

# run() 인터페이스로 호출
print('\n📋 run() 인터페이스 테스트:')
result = checker_agent.run('이지은, 2학년, 컴퓨터공학과(공과대학), 파이썬 기초 이수')
print(result)
print('\n→ exercise 3에서 이 인터페이스를 통해 에이전트를 오케스트레이터에 연결합니다.')

✅ BaseAgentWorker 클래스 정의 완료
✅ EligibilityCheckerAgent 클래스 정의 완료

생성된 에이전트: EligibilityCheckerAgent(name="Eligibility Checker", task="Task 1b")
  name: Eligibility Checker
  task_id: Task 1b

📋 run() 인터페이스 테스트:
## AI 동아리 신입 부원 자격 검사 결과

**지원자:** 이지은

---

### 1. 학년 기준:

*   **결과:** ✅ 합격
*   **설명:** 2학년으로, 1~3학년 지원 가능 기준을 충족합니다.

### 2. 전공 기준:

*   **결과:** ✅ 합격
*   **설명:** 컴퓨터공학과(공과대학) 소속으로, 공과대학 또는 자연과학대학 소속 기준을 충족합니다.

### 3. 이수 과목 기준:

*   **결과:** ✅ 합격
*   **설명:** '파이썬 기초'를 이수 완료하여, '파이썬 기초' 또는 '선형대수' 중 최소 1개 이수 기준을 충족합니다.

---

### 최종 판정:

**PASS**

---

→ Week 3에서 이 인터페이스를 통해 에이전트를 오케스트레이터에 연결합니다.
